In [ ]:
import os
import json
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

import sys
sys.path.append('.../data')

from hdbscan import HDBSCAN

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.stats import pearsonr

from utils.functions import load_yaml
from utils.graphfunction import load_graph, get_uniprot_from_nodes, get_res_from_nodes
from data.augmentation import ss_aug
from data.scaler import sin_cos_transform, pca_transform
from data.neighborExtractor import extract_neighbor_features, extract_neighbor_features_parallel

from data.vizutils import plot_cluster_distribution

from models.RRGCL.src.mutprofile.similarity import sim_pair_node

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Configuration
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

# Load Graph Data
# finalG = load_graph((f"{DATABASE}/cleaned_weighted_graph_041226.pkl"))

# Feature DataFrame
org_res_feat_df = pd.read_csv(f'{config.DATABASE}/merged_feature_data_v041226.csv')
# org_ngb_feat_df = pd.read_csv('../data/proc_data/ngb_weighted_mean_feat.csv')
# org_ngb_feat_df = pd.read_csv('../data/proc_data/ngb_orthogonal_proj_feat.csv')

# 1. Clustering

## 1-1 Data Ready (NaN Processing)

In [ ]:
ss_cols = ['rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_accessibility', 'dssp_TCO', 'dssp_alpha', 'dssp_phi', 'dssp_psi']

In [ ]:
aa1_cols = [col for col in org_res_feat_df.columns if col.startswith('aa1')]
aa1_cols.remove('aa1')

In [ ]:
res_AA1 = org_res_feat_df[['node_id']+aa1_cols].copy()
ngb_AA1 = org_ngb_feat_df[['node_id']+aa1_cols].copy()

In [ ]:
org_res_feat_df.head(3)

In [ ]:
proc_res_feat_df = pd.merge(org_res_feat_df, res_AA1, on='node_id', how='right')
proc_res_feat_df

In [ ]:
cluster_cols = ['node_id',
                'aa1_KYTJ820101', 'aa1_KLEP840101', 'aa1_BHAR880101', 'aa1_JANJ780101', 'aa1_CHOP780201', 'aa1_GRAR740102', 'aa1_GRAR740103',
                'rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_accessibility', 'dssp_TCO', 'dssp_alpha', 'dssp_phi', 'dssp_psi',
                # 'hmm_neff', 'pssm_entropy'
                ]

hmm_cols = [col for col in org_res_feat_df.columns if col.startswith('hmm')]
pssm_cols = [col for col in org_res_feat_df.columns if col.startswith('pssm')]

ngb_cluster_cols = cluster_cols.copy() # + ['ss_helix', 'ss_sheet', 'ss_loop']

cluster_res_feat_df = org_res_feat_df[cluster_cols]
cluster_ngb_feat_df = org_ngb_feat_df[ngb_cluster_cols]

In [ ]:
cluster_ngb_feat_df

In [ ]:
aug_cluster_res_feat_df = ss_aug(cluster_res_feat_df, ss_cols, mode='neighbor', graph=finalG, num_ref=5)
# aug_cluster_ngb_feat_df = ss_aug(cluster_ngb_feat_df, ss_cols+['ss_helix', 'ss_sheet', 'ss_loop'], mode='neighbor', graph=finalG, num_ref=5)
aug_cluster_ngb_feat_df = ss_aug(cluster_ngb_feat_df, ss_cols, mode='neighbor', graph=finalG, num_ref=5)

# Residue DataFrame
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_phi', 'sin')
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_phi', 'cos')

aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_psi', 'sin')
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_psi', 'cos')

aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_alpha', 'sin')
aug_cluster_res_feat_df = sin_cos_transform(aug_cluster_res_feat_df, 'dssp_alpha', 'cos')

# Neighbor DataFrame
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_phi', 'sin')
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_phi', 'cos')

aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_psi', 'sin')
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_psi', 'cos')

aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_alpha', 'sin')
aug_cluster_ngb_feat_df = sin_cos_transform(aug_cluster_ngb_feat_df, 'dssp_alpha', 'cos')

In [ ]:
corr = aug_cluster_res_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Residue] Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
corr = aug_cluster_ngb_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Neighbor] Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
len(aa1_cols)

In [ ]:
aa1_cols = [col for col in aug_cluster_ngb_feat_df if col.startswith('aa1')]

fig, axs = plt.subplots(2,4, figsize=(15,4))
axs = axs.ravel()

for i, col in enumerate(aa1_cols):
    axs[i].hist(aug_cluster_ngb_feat_df[col], bins=50)
    axs[i].set_title(f"{col} || ({len(aug_cluster_ngb_feat_df[col].unique())})")
    axs[i].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# aug_cluster_res_feat_df.to_csv('../data/proc_data/aug_res_feature_data_for_clustering_wm.csv', index=False)
# aug_cluster_ngb_feat_df.to_csv('../data/proc_data/aug_ngb_wm_feature_data_for_clustering.csv', index=False)

## 1-2 Feature Engineering for Clustering

### AAIndex1

In [ ]:
reS_scaler = StandardScaler()
ngb_scaler = StandardScaler()
res_AA1[aa1_cols] = reS_scaler.fit_transform(res_AA1[aa1_cols])
ngb_AA1[aa1_cols] = ngb_scaler.fit_transform(ngb_AA1[aa1_cols])

In [ ]:
res_pca_aa1_df, res_pca = pca_transform(res_AA1, aa1_cols, 2)
ngb_pca_aa1_df, ngb_pca = pca_transform(ngb_AA1, aa1_cols, 2)
print("[Residue] AAindex Explained variance:", res_pca.explained_variance_ratio_, res_pca.explained_variance_ratio_.sum())
print("[Neighbor] AAindex Explained variance:", ngb_pca.explained_variance_ratio_, ngb_pca.explained_variance_ratio_.sum())

In [ ]:
proc_cluster_res_feat_df = aug_cluster_res_feat_df.drop(aa1_cols, axis=1)
proc_cluster_ngb_feat_df = aug_cluster_ngb_feat_df.drop(aa1_cols, axis=1)

proc_cluster_res_feat_df = pd.merge(proc_cluster_res_feat_df, res_pca_aa1_df, on='node_id', how='left')
proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_aa1_df, on='node_id', how='left')

### Secondary Strcture

In [ ]:
ss_pca_cols1 = ['rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_accessibility']
ss_pca_cols2 = ['dssp_TCO', 'dssp_phi_sin', 'dssp_phi_cos', 'dssp_psi_sin', 'dssp_psi_cos', 'dssp_alpha_sin', 'dssp_alpha_cos']

all_ss_pca_cols = ss_pca_cols1 + ss_pca_cols2

In [ ]:
reS_scaler = StandardScaler()
ngb_scaler = StandardScaler()
proc_cluster_res_feat_df[all_ss_pca_cols] = reS_scaler.fit_transform(proc_cluster_res_feat_df[all_ss_pca_cols])
proc_cluster_ngb_feat_df[all_ss_pca_cols] = ngb_scaler.fit_transform(proc_cluster_ngb_feat_df[all_ss_pca_cols])

In [ ]:
res_pca_ss_df, ss_res_pca = pca_transform(proc_cluster_res_feat_df, ss_pca_cols1, 2)
ngb_pca_ss_df, ss_ngb_pca = pca_transform(proc_cluster_ngb_feat_df, ss_pca_cols1, 2)
print("[Residue] ss1 Explained variance:", ss_res_pca.explained_variance_ratio_, ss_res_pca.explained_variance_ratio_.sum())
print("[Neighbor] ss1 Explained variance:", ss_ngb_pca.explained_variance_ratio_, ss_ngb_pca.explained_variance_ratio_.sum())

In [ ]:
res_pca_ss_df2, ss_res_pca2 = pca_transform(proc_cluster_res_feat_df, ss_pca_cols2, 2)
ngb_pca_ss_df2, ss_ngb_pca2 = pca_transform(proc_cluster_ngb_feat_df, ss_pca_cols2, 2)
print("[Residue] ss2 Explained variance:", ss_res_pca2.explained_variance_ratio_, ss_res_pca2.explained_variance_ratio_.sum())
print("[Neighbor] ss2 Explained variance:", ss_ngb_pca2.explained_variance_ratio_, ss_ngb_pca2.explained_variance_ratio_.sum())

In [ ]:
# ngb_pca_ss_df3, ss_ngb_pca3 = pca_transform(proc_cluster_ngb_feat_df, ['ss_helix', 'ss_sheet', 'ss_loop'], 1)
# print("[Neighbor] ss2 Explained variance:", ss_ngb_pca3.explained_variance_ratio_, ss_ngb_pca3.explained_variance_ratio_.sum())

In [ ]:
proc_cluster_res_feat_df = proc_cluster_res_feat_df.drop(all_ss_pca_cols+['dssp_alpha', 'dssp_phi', 'dssp_psi'], axis=1)
# proc_cluster_ngb_feat_df = proc_cluster_ngb_feat_df.drop(all_ss_pca_cols+['dssp_alpha', 'dssp_phi', 'dssp_psi']+['ss_helix', 'ss_sheet', 'ss_loop'], axis=1)
proc_cluster_ngb_feat_df = proc_cluster_ngb_feat_df.drop(all_ss_pca_cols+['dssp_alpha', 'dssp_phi', 'dssp_psi'], axis=1)

proc_cluster_res_feat_df = pd.merge(proc_cluster_res_feat_df, res_pca_ss_df, on='node_id', how='left')
proc_cluster_res_feat_df = pd.merge(proc_cluster_res_feat_df, res_pca_ss_df2, on='node_id', how='left')

proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_ss_df, on='node_id', how='left')
proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_ss_df2, on='node_id', how='left')

proc_cluster_ngb_feat_df = pd.merge(proc_cluster_ngb_feat_df, ngb_pca_ss_df3, on='node_id', how='left')

In [ ]:
corr = proc_cluster_res_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Residue] Proc Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
corr = proc_cluster_ngb_feat_df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('[Neighbor] Proc Features Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
data = proc_cluster_res_feat_df.select_dtypes(include=['number']).sample(5000, random_state=42)
scaled_data = StandardScaler().fit_transform(data)

tsne_params = {
    'n_components': 2,
    'perplexity': 30,
    'random_state': 42,
    'n_jobs': -1
}

projection = TSNE(**tsne_params).fit_transform(scaled_data)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=projection[:, 0], 
    y=projection[:, 1], 
    s=2,
    alpha=0.3,
    linewidth=0
)
plt.show()

In [ ]:
data = proc_cluster_ngb_feat_df.select_dtypes(include=['number']).sample(5000, random_state=42)
scaled_data = StandardScaler().fit_transform(data)

tsne_params = {
    'n_components': 2,
    'perplexity': 30,
    'random_state': 42,
    'n_jobs': -1
}

projection = TSNE(**tsne_params).fit_transform(scaled_data)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=projection[:, 0], 
    y=projection[:, 1], 
    s=2,
    alpha=0.3,
    linewidth=0
)
plt.show()

## Clustering

In [ ]:
from sklearn.mixture import GaussianMixture

In [ ]:
res_feat = proc_cluster_res_feat_df.select_dtypes(include=['number'])
ngb_feat = proc_cluster_ngb_feat_df.select_dtypes(include=['number'])

In [ ]:
# res_clusterer = hdbscan.HDBSCAN(cluster_selection_epsilon=eps,
#                                 min_cluster_size=mcs,
#                                 min_samples=mss,
#                                 cluster_selection_method=csm,)

In [ ]:
res_gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
res_gmm.fit(res_feat.to_numpy())
labels = res_gmm.predict(res_feat)
proc_cluster_res_feat_df['cluster'] = labels

In [ ]:
ngb_gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
ngb_gmm.fit(ngb_feat.to_numpy())
labels = ngb_gmm.predict(ngb_feat)
proc_cluster_ngb_feat_df['cluster'] = labels

In [ ]:
models = []
# n_components = range(5, 11, 1)
n_components = [100, 300]
print(list(n_components))

for n in n_components:
    print("N Components", n)
    models.append(GaussianMixture(n, covariance_type='full', random_state=42).fit(res_feat))

# BIC 점수 시각화 (낮을수록 좋은 모델)
plt.plot(n_components, [m.bic(res_feat) for m in models], label='BIC')
plt.plot(n_components, [m.aic(res_feat) for m in models], label='AIC')
plt.legend(loc='best')
plt.xlabel('n_components')
plt.show()

In [ ]:
ngb_models = []
n_components = range(7, 16, 1)
print(list(n_components))

for n in n_components:
    print("N Components", n)
    ngb_models.append(GaussianMixture(n, covariance_type='full', random_state=42).fit(ngb_feat))

# BIC 점수 시각화 (낮을수록 좋은 모델)
plt.plot(n_components, [m.bic(ngb_feat) for m in ngb_models], label='BIC')
plt.plot(n_components, [m.aic(ngb_feat) for m in ngb_models], label='AIC')
plt.legend(loc='best')
plt.xlabel('n_components')
plt.show()

In [ ]:
plt.savefig('neighbor_GMCluster.png', )

## Cluster Analysis

In [ ]:
# res_cluster_results = pd.read_csv('')
# ngb_cluster_results = pd.read_csv('')

In [ ]:
# Add mapping features
proc_cluster_res_feat_df['residue'] = proc_cluster_res_feat_df['node_id'].map(get_res_from_nodes)
proc_cluster_res_feat_df['uniprot'] = proc_cluster_res_feat_df['node_id'].map(get_uniprot_from_nodes)

proc_cluster_ngb_feat_df['residue'] = proc_cluster_ngb_feat_df['node_id'].map(get_res_from_nodes)
proc_cluster_ngb_feat_df['uniprot'] = proc_cluster_ngb_feat_df['node_id'].map(get_uniprot_from_nodes)

In [ ]:
print("Residue-Level Clustering: Residue Types")
plot_cluster_distribution(proc_cluster_res_feat_df, 'residue', grid_rows=3, grid_cols=4)

print("Neighbor-Level Clustering: Residue Types")
plot_cluster_distribution(proc_cluster_ngb_feat_df, 'residue', grid_rows=3, grid_cols=4)

# 2. Similarity

In [ ]:
hmm_feat = ['node_id', 'aa1'] + [col for col in org_res_feat_df.columns if col.startswith('hmm')]
pssm_feat = ['node_id', 'aa1'] + [col for col in org_res_feat_df.columns if col.startswith('pssm')]
am_feat = ['node_id', 'aa1'] + [col for col in org_res_feat_df.columns if col.startswith('am')]

In [ ]:
hmm_df = org_res_feat_df[hmm_feat]
pssm_df = org_res_feat_df[pssm_feat]
am_df = org_res_feat_df[am_feat]

In [ ]:
am_df

In [ ]:
node1 = 'a8mt69_14_leu'
node2 = 'q9y6q9_1086_gln'

In [ ]:
am_sim = sim_pair_node(org_res_feat_df.set_index('node_id'), node1, node2, feat_prefix='am', sim_method='pearson')
pssm_sim = sim_pair_node(org_res_feat_df.set_index('node_id') , node1, node2, feat_prefix='pssm', sim_method='entropy_corr')
hmm_sim = sim_pair_node(org_res_feat_df.set_index('node_id') , node1, node2, feat_prefix='hmm', sim_method='es_corr')

print(am_sim, pssm_sim, hmm_sim)

# 3. Similarity Analysis with Clustering Results

In [ ]:
ngb_cluster_df = pd.read_csv('../neighbor_cluster_labels_eps0_mcs9_mss5_eom.csv')
res_cluster_df = pd.read_csv('../residue_cluster_labels_eps0_mcs19_mss5_eom.csv')
res_cluster_df.rename({'cluster_label': 'cluster'}, axis=1, inplace=True)
ngb_cluster_df.rename({'cluster_label': 'cluster'}, axis=1, inplace=True)

In [ ]:
# Add mapping features
ngb_cluster_df['residue'] = ngb_cluster_df['node_id'].map(get_res_from_nodes)
ngb_cluster_df['uniprot'] = ngb_cluster_df['node_id'].map(get_uniprot_from_nodes)

res_cluster_df['residue'] = res_cluster_df['node_id'].map(get_res_from_nodes)
res_cluster_df['uniprot'] = res_cluster_df['node_id'].map(get_uniprot_from_nodes)

In [ ]:
print("Residue-Level Clustering: Residue Types")
plot_cluster_distribution(res_cluster_df, 'residue', grid_rows=10, grid_cols=5)

print("Neighbor-Level Clustering: Residue Types")
plot_cluster_distribution(ngb_cluster_df, 'residue', grid_rows=2, grid_cols=3)

In [ ]:
def calc_batch_sim(pair_list, indexed_df, feat_prefix, sim_method):
    if not pair_list:
        return []
    
    sims = []
    for node1, node2 in pair_list:
        sim = sim_pair_node(indexed_df, node1, node2, feat_prefix=feat_prefix, sim_method=sim_method)
        sims.append(sim)
    return sims

In [ ]:
indexed_df = org_res_feat_df.set_index('node_id')
n_jobs_actual = 4 

fig, axs = plt.subplots(13, 4, figsize=(12, 32))
axs = axs.ravel()

res_cluster_idxs = sorted([c for c in res_cluster_df.cluster.unique()])

for i, cls_idx in enumerate(tqdm(res_cluster_idxs, desc="Processing Res Clusters")):
    if i >= len(axs): break
    
    temp = res_cluster_df[res_cluster_df['cluster'] == cls_idx]
    node_lists = list(temp['node_id'])
    
    if len(node_lists) < 2:
        continue

    pairs = [(node_lists[j], node_lists[j+1]) for j in range(len(node_lists) - 1)]
    pair_chunks = [chunk.tolist() for chunk in np.array_split(pairs, n_jobs_actual) if len(chunk) > 0]

    am_sim_list_nested = Parallel(n_jobs=n_jobs_actual)(
        delayed(calc_batch_sim)(chunk, indexed_df, 'am', 'pearson') 
        for chunk in pair_chunks
    )

    pssm_sim_list_nested = Parallel(n_jobs=n_jobs_actual)(
        delayed(calc_batch_sim)(chunk, indexed_df, 'pssm', 'entropy_corr') 
        for chunk in pair_chunks
    )
    
    am_sim_list = [sim for sublist in am_sim_list_nested for sim in sublist]
    pssm_sim_list = [sim for sublist in pssm_sim_list_nested for sim in sublist]

    ax = axs[i]
    
    x = np.array(am_sim_list)
    y = np.array(pssm_sim_list)
    
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    x_valid, y_valid = x[valid_mask], y[valid_mask]
    
    if len(x_valid) > 1:
        ax.scatter(x_valid, y_valid, alpha=0.6, s=15, edgecolors='none')
        
        m, b = np.polyfit(x_valid, y_valid, 1)
        ax.plot(x_valid, m * x_valid + b, color='red', linewidth=1.5)
        ax.plot([-1.2, 1.2], [-1.2, 1.2], color='black', linestyle='--', alpha=0.3, linewidth=1)
        
        r_value, _ = pearsonr(x_valid, y_valid)
        
        textstr = f'r = {r_value:.2f}'
        props = dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor='gray')
        
        ax.text(0.05, 0.90, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)

    ax.set_title(f'Cluster: {cls_idx}')
    ax.set_xlabel('AM Similarity')
    ax.set_ylabel('PSSM Similarity')
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)

for j in range(i + 1, len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
indexed_df = org_res_feat_df.set_index('node_id')
n_jobs_actual = 4 

fig, axs = plt.subplots(2, 3, figsize=(12, 8))
axs = axs.ravel()

ngb_cluster_idxs = sorted([c for c in ngb_cluster_df.cluster.unique()])

for i, cls_idx in enumerate(tqdm(ngb_cluster_idxs, desc="Processing Res Clusters")):
    if i >= len(axs): break
    
    temp = ngb_cluster_df[ngb_cluster_df['cluster'] == cls_idx]
    node_lists = list(temp['node_id'])
    
    if len(node_lists) < 2:
        continue

    pairs = [(node_lists[j], node_lists[j+1]) for j in range(len(node_lists) - 1)]
    pair_chunks = [chunk.tolist() for chunk in np.array_split(pairs, n_jobs_actual) if len(chunk) > 0]

    am_sim_list_nested = Parallel(n_jobs=n_jobs_actual)(
        delayed(calc_batch_sim)(chunk, indexed_df, 'am', 'pearson') 
        for chunk in pair_chunks
    )

    pssm_sim_list_nested = Parallel(n_jobs=n_jobs_actual)(
        delayed(calc_batch_sim)(chunk, indexed_df, 'pssm', 'entropy_corr') 
        for chunk in pair_chunks
    )
    
    am_sim_list = [sim for sublist in am_sim_list_nested for sim in sublist]
    pssm_sim_list = [sim for sublist in pssm_sim_list_nested for sim in sublist]

    ax = axs[i]
    
    x = np.array(am_sim_list)
    y = np.array(pssm_sim_list)
    
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    x_valid, y_valid = x[valid_mask], y[valid_mask]
    
    if len(x_valid) > 1:
        ax.scatter(x_valid, y_valid, alpha=0.6, s=15, edgecolors='none')
        
        m, b = np.polyfit(x_valid, y_valid, 1)
        ax.plot(x_valid, m * x_valid + b, color='red', linewidth=1.5)
        ax.plot([-1.2, 1.2], [-1.2, 1.2], color='black', linestyle='--', alpha=0.3, linewidth=1)
        
        r_value, _ = pearsonr(x_valid, y_valid)
        
        textstr = f'r = {r_value:.2f}'
        props = dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor='gray')
        
        ax.text(0.60, 0.85, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)

    ax.set_title(f'Cluster: {cls_idx}')
    ax.set_xlabel('AM Similarity')
    ax.set_ylabel('PSSM Similarity')
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)

for j in range(i + 1, len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# import random
# import itertools
# from tqdm.autonotebook import tqdm

# res_cluster_idxs = list(res_cluster_df.cluster.unique())
# res_cluster_idxs.sort()
# fig, axs = plt.subplots(10, 5, figsize=(15, 20))
# axs = axs.ravel()

# n_sample = 10_000

# for i, cls_idx in enumerate(res_cluster_idxs):
#     if cls_idx == -1:
#         continue
#     print('Cluster', cls_idx)
#     temp = res_cluster_df[res_cluster_df['cluster']==cls_idx]
#     sim_list = []
#     node_lists = list(temp['node_id'])

#     for _ in tqdm(range(n_sample)):
#         node1, node2 = random.sample(node_lists, 2)
#         sim_list.append(sim_pair_node(org_res_feat_df.set_index('node_id'), node1, node2, feat_prefix='am', sim_method='pearson'))

#     axs[i].hist(sim_list)

# plt.tight_layout()
# plt.show()

In [ ]:
import random
from tqdm.autonotebook import tqdm
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

indexed_df = org_res_feat_df.set_index('node_id')

def calc_single_sim(node_list, indexed_df):
    node1, node2 = random.sample(node_list, 2)
    return sim_pair_node(indexed_df, node1, node2, feat_prefix='am', sim_method='pearson')

res_cluster_idxs = sorted([c for c in res_cluster_df.cluster.unique() if c != -1])
n_sample = 10_000

fig, axs = plt.subplots(10, 5, figsize=(15, 20))
axs = axs.ravel()

for i, cls_idx in enumerate(tqdm(res_cluster_idxs, desc="Processing Clusters")):
    if i >= len(axs): break
    
    temp = res_cluster_df[res_cluster_df['cluster'] == cls_idx]
    node_lists = list(temp['node_id'])
    
    sim_list = Parallel(n_jobs=12)(
        delayed(calc_single_sim)(node_lists, indexed_df) for _ in range(n_sample)
    )
    
    axs[i].hist(sim_list, bins=30, alpha=0.7)
    axs[i].set_title(f'Cluster {cls_idx}')


plt.tight_layout()
plt.show()

In [ ]:
nbh_cluster_idxs = sorted([c for c in nbh_cluster_df.cluster.unique() if c != -1])
n_sample = 10_000

fig, axs = plt.subplots(10, 5, figsize=(15, 20))
axs = axs.ravel()

for i, cls_idx in enumerate(tqdm(nbh_cluster_idxs, desc="Processing Clusters")):
    if i >= len(axs): break
    
    temp = res_cluster_df[res_cluster_df['cluster'] == cls_idx]
    node_lists = list(temp['node_id'])
    
    sim_list = Parallel(n_jobs=12)(
        delayed(calc_single_sim)(node_lists, indexed_df) for _ in range(n_sample)
    )
    
    axs[i].hist(sim_list, bins=30, alpha=0.7)
    axs[i].set_title(f'Cluster {cls_idx}')